In [2]:
import math

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

## 01 Common Data

In [8]:
models = ['miroc6', 'ec_earth3_veg_lr' , 'access_esm1_5', 'ipsl_cm6a_lr', 'mpi_esm1_2_hr'] #['miroc6', 'ec_earth3_veg_lr' , 'access_esm1_5', 'ipsl_cm6a_lr', 'mpi_esm1_2_hr']
models = ['ec_earth3_veg_lr']
scenarios = ['historical', 'ssp245', 'ssp585']
colors = ['black', 'cornflowerblue', 'coral']

## 02 MSLP

#### 02.01 MSLP TimeSeries

In [21]:
n_pcs = 10

for model in models:

    print(f'Model: {model}')

    fig, axes = plt.subplots(n_pcs, 1, figsize=(10, 2*n_pcs))
    axes = axes.flatten()

    #### ERA5

    pcs_era5 = pd.read_csv(f'outputs/era5/era5_mslp_pcs.csv', index_col=0, parse_dates=True)

    for i in range(n_pcs):

        x = pcs_era5.index.values
        y = pcs_era5[f'PC{i+1}'].values

        axes[i].plot(x, y, color = 'darkgray', label='ERA5')
        axes[i].set_title(f'PC {i+1}')

    #### GCM
    for s, scenario in enumerate(scenarios):

        pcs_mslp = pd.read_csv(f'outputs/cmip6_models/{model}/{scenario}/mslp_{model}_{scenario}_pcs_95.csv', index_col=0, parse_dates=True)

        for i in range(n_pcs):
            x = pcs_mslp.index.values
            y = pcs_mslp[f'PC{i+1}'].values
            axes[i].plot(x, y, color=colors[s], label=f'GCM - {scenario}')

        # --- Create ONE common legend ---
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))

    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc='upper center',
        ncol=len(by_label),
        frameon=False
    )

    plt.suptitle(f'{model} MSLP Time SeriesPCs', y=0.98)        
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.savefig(f'outputs/figures/bias_corr/{model}_mslp_pcs_time_series.png', dpi=300)
    plt.close()

Model: miroc6
Model: ec_earth3_veg_lr
Model: access_esm1_5
Model: ipsl_cm6a_lr
Model: mpi_esm1_2_hr


#### 02.02 Histograms of PCs

In [22]:
n_pcs = 10

for model in models:
    print(f'Model: {model}')

    n_cols = 5
    n_rows = math.ceil(n_pcs / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*5))
    axes = axes.flatten()

    #### ERA5
    pcs_era5 = pd.read_csv(f'outputs/era5/era5_mslp_pcs.csv',index_col=0,parse_dates=True)

    #### GCM 
    pcs_gcm_all = {}
    for scenario in scenarios:
        pcs_gcm_all[scenario] = pd.read_csv(f'outputs/cmip6_models/{model}/{scenario}/mslp_{model}_{scenario}_pcs_95.csv',index_col=0,parse_dates=True)

    for i in range(n_pcs):

        ax = axes[i]
        data_all = []
        era5_vals = pcs_era5[f'PC{i+1}'].values
        data_all.append(era5_vals)
        for scenario in scenarios:
            vals = pcs_gcm_all[scenario][f'PC{i+1}'].values
            data_all.append(vals)

        combined = np.concatenate(data_all)
        bins = np.histogram_bin_edges(combined, bins=30) 

        # --- Plot ERA5 ---
        ax.hist(era5_vals, bins=bins, color='darkgray', alpha=0.5, label='ERA5', density=True)

        # --- Plot GCMs ---
        for s, scenario in enumerate(scenarios):
            vals = pcs_gcm_all[scenario][f'PC{i+1}'].values
            ax.hist(vals, bins=bins, color=colors[s], alpha=0.5, label=scenario, density=True)

        ax.set_title(f'PC {i+1}')

    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc='upper center',
        ncol=len(by_label),
        frameon=False
    )

    plt.suptitle(f'{model} MSLP Histogram PCs', y=0.95)        
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'outputs/figures/bias_corr/{model}_mslp_pcs_histograms.png', dpi=300)
    plt.close()

Model: miroc6
Model: ec_earth3_veg_lr
Model: access_esm1_5
Model: ipsl_cm6a_lr
Model: mpi_esm1_2_hr


#### 02.03. Corr Matrix

In [23]:
for model in models:
    print(f'Model: {model}')

    fig, axes = plt.subplots(1, 4, figsize=(4*5, 1*6))

    #### ERA5
    pcs_era5 = pd.read_csv(f'outputs/era5/era5_mslp_pcs.csv',index_col=0,parse_dates=True)
    im = sns.heatmap(pcs_era5.corr(), ax = axes[0], cmap='cool', vmin=0,vmax=0.3,cbar=False)
    axes[0].set_title(f'ERA5')

    #### GCM
    for s, scenario in enumerate(scenarios):
        pcs_gmc = pd.read_csv(f'outputs/cmip6_models/{model}/{scenario}/mslp_{model}_{scenario}_pcs_95.csv',index_col=0,parse_dates=True)
        sns.heatmap(pcs_gmc.corr(), ax = axes[s+1], cmap='cool', vmin=0,vmax=0.3,cbar=False)
        axes[s+1].set_title(f'{scenario}')

    # --- Create ONE shared colorbar ---
    cbar = fig.colorbar(im.collections[0],ax=axes,orientation='horizontal', fraction=0.05, pad=0.01)
    cbar.set_label('Correlation')

    plt.tight_layout()
    plt.suptitle(f'{model} MSLP PCs Correlation', y=0.98)
    plt.tight_layout(rect=[0, 0.15, 1, 0.98])
    plt.savefig(f'outputs/figures/bias_corr/{model}_mslp_pcs_correlation.png', dpi=300)
    plt.close()

Model: miroc6


/tmp/ipykernel_673548/220730467.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_673548/220730467.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 1, 0.98])


Model: ec_earth3_veg_lr


/tmp/ipykernel_673548/220730467.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_673548/220730467.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 1, 0.98])


Model: access_esm1_5


/tmp/ipykernel_673548/220730467.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_673548/220730467.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 1, 0.98])


Model: ipsl_cm6a_lr


/tmp/ipykernel_673548/220730467.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_673548/220730467.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 1, 0.98])


Model: mpi_esm1_2_hr


/tmp/ipykernel_673548/220730467.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_673548/220730467.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 1, 0.98])


## 03 SST

#### 02.01 SST TimeSeries

In [5]:
var = 'index'

In [6]:
pcs_era5 = pd.read_csv(f'outputs/era5/era5_{var}_pcs.csv', index_col=0, parse_dates=True)
n_pcs = pcs_era5.shape[1]


In [9]:
for model in models:

    print(f'Model: {model}')

    fig, axes = plt.subplots(n_pcs, 1, figsize=(10, 2*n_pcs))
    axes = axes.flatten()

    #### ERA5

    pcs_era5 = pd.read_csv(f'outputs/era5/era5_{var}_pcs.csv', index_col=0, parse_dates=True)

    for i in range(n_pcs):

        x = pcs_era5.index.values
        y = pcs_era5[f'PC{i+1}'].values

        axes[i].plot(x, y, color = 'darkgray', label='ERA5')
        axes[i].set_title(f'PC {i+1}')

    #### GCM
    for s, scenario in enumerate(scenarios):

        pcs_sst = pd.read_csv(f'outputs/cmip6_models/{model}/{scenario}/{var}_{model}_{scenario}_pcs.csv', index_col=0, parse_dates=True)

        for i in range(n_pcs):
            x = pcs_sst.index.values
            y = pcs_sst[f'PC{i+1}'].values
            axes[i].plot(x, y, color=colors[s], label=f'GCM - {scenario}')

        # --- Create ONE common legend ---
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))

    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc='upper center',
        ncol=len(by_label),
        frameon=False
    )

    plt.suptitle(f'{model} {var} Time SeriesPCs', y=0.98)        
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.savefig(f'outputs/figures/bias_corr/{model}_{var}_pcs_time_series.png', dpi=300)
    plt.close()

Model: ec_earth3_veg_lr


#### 02.02 Histograms of PCs

In [ ]:
for model in models:
    print(f'Model: {model}')

    n_cols = n_pcs
    n_rows = 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*5))
    axes = axes.flatten()

    #### ERA5
    pcs_era5 = pd.read_csv(f'outputs/era5/era5_sst_pcs.csv',index_col=0,parse_dates=True)

    #### GCM 
    pcs_gcm_all = {}
    for scenario in scenarios:
        pcs_gcm_all[scenario] = pd.read_csv(f'outputs/cmip6_models/{model}/{scenario}/sst_{model}_{scenario}_pcs.csv',index_col=0,parse_dates=True)

    for i in range(n_pcs):

        ax = axes[i]
        data_all = []
        era5_vals = pcs_era5[f'PC{i+1}'].values
        data_all.append(era5_vals)
        for scenario in scenarios:
            vals = pcs_gcm_all[scenario][f'PC{i+1}'].values
            data_all.append(vals)

        combined = np.concatenate(data_all)
        bins = np.histogram_bin_edges(combined, bins=30) 

        # --- Plot ERA5 ---
        ax.hist(era5_vals, bins=bins, color='darkgray', alpha=0.5, label='ERA5', density=True)

        # --- Plot GCMs ---
        for s, scenario in enumerate(scenarios):
            vals = pcs_gcm_all[scenario][f'PC{i+1}'].values
            ax.hist(vals, bins=bins, color=colors[s], alpha=0.5, label=scenario, density=True)

        ax.set_title(f'PC {i+1}')

    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc='upper center',
        ncol=len(by_label),
        frameon=False
    )

    plt.suptitle(f'{model} SST Histogram PCs', y=0.94)        
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'outputs/figures/bias_corr/{model}_sst_pcs_histograms.png', dpi=300)
    plt.close()

Model: miroc6


#### 02.03 Corr Matrix

In [ ]:
for model in models:
    print(f'Model: {model}')

    fig, axes = plt.subplots(1, 4, figsize=(4*5, 1*6))

    #### ERA5
    pcs_era5 = pd.read_csv(f'outputs/era5/era5_sst_pcs.csv',index_col=0,parse_dates=True)
    im = sns.heatmap(pcs_era5.corr(), ax = axes[0], cmap='cool', vmin=0,vmax=0.3,cbar=False)
    axes[0].set_title(f'ERA5')

    #### GCM
    for s, scenario in enumerate(scenarios):
        pcs_gmc = pd.read_csv(f'outputs/cmip6_models/{model}/{scenario}/sst_{model}_{scenario}_pcs.csv',index_col=0,parse_dates=True)
        sns.heatmap(pcs_gmc.corr(), ax = axes[s+1], cmap='cool', vmin=0,vmax=0.3,cbar=False)
        axes[s+1].set_title(f'{scenario}')

    # --- Create ONE shared colorbar ---
    cbar = fig.colorbar(im.collections[0],ax=axes,orientation='horizontal', fraction=0.05, pad=0.01)
    cbar.set_label('Correlation')

    plt.tight_layout()
    plt.suptitle(f'{model} SST PCs Correlation', y=0.98)
    plt.tight_layout(rect=[0, 0.15, 1, 0.98])
    plt.savefig(f'outputs/figures/bias_corr/{model}_sst_pcs_correlation.png', dpi=300)
    plt.close()

Model: miroc6


/tmp/ipykernel_3307808/3246405045.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_3307808/3246405045.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 1, 0.98])
